In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import DeltaTable

df = spark.read.table("db_catalog.silver.orders_silver")
df.display()

order_id,customer_id,product_id,order_date,quantity,total_amount,unit_price,customer_avg_order_value,ptg_of_customer_spend_on_order
O00186,C00001,P0126,2023-02-18,1,1431.12,1431.12,3315.17,10.79
O02770,C00001,P0019,2023-12-09,2,461.12,230.56,3315.17,3.48
O05531,C00001,P0456,2023-09-10,5,9524.35,1904.87,3315.17,71.82
O08359,C00001,P0338,2023-03-09,1,1844.1,1844.1,3315.17,13.91
O00775,C00002,P0323,2024-01-16,1,1248.81,1248.81,2157.11,7.24
O02495,C00002,P0354,2023-05-09,4,1158.68,289.67,2157.11,6.71
O04328,C00002,P0155,2024-07-18,5,61.75,12.35,2157.11,0.36
O04879,C00002,P0139,2023-06-20,3,1197.09,399.03,2157.11,6.94
O05231,C00002,P0133,2023-05-18,3,646.89,215.63,2157.11,3.75
O05319,C00002,P0028,2023-05-12,4,6731.64,1682.91,2157.11,39.01


In [0]:
df_dimcus = spark.sql("select dim_customer_key, customer_id as dim_customer_id from db_catalog.gold.dim_customers")

df_dimpro = spark.sql("""select dim_product_key, product_id as dim_product_id from db_catalog.gold.dim_products WHERE current_flag = true""")


In [0]:
df_fact = df.join(df_dimcus, df['customer_id'] == df_dimcus['dim_customer_id'],how='left').join(df_dimpro, df['product_id'] == df_dimpro['dim_product_id'],how='left')


In [0]:
df_fact_new = df_fact.drop('dim_customer_id','dim_product_id','customer_id','product_id')
df_fact_new.display()

order_id,order_date,quantity,total_amount,unit_price,customer_avg_order_value,ptg_of_customer_spend_on_order,dim_customer_key,dim_product_key
O00186,2023-02-18,1,1431.12,1431.12,3315.17,10.79,1,126
O02770,2023-12-09,2,461.12,230.56,3315.17,3.48,1,19
O05531,2023-09-10,5,9524.35,1904.87,3315.17,71.82,1,456
O08359,2023-03-09,1,1844.1,1844.1,3315.17,13.91,1,338
O00775,2024-01-16,1,1248.81,1248.81,2157.11,7.24,2,323
O02495,2023-05-09,4,1158.68,289.67,2157.11,6.71,2,354
O04328,2024-07-18,5,61.75,12.35,2157.11,0.36,2,155
O04879,2023-06-20,3,1197.09,399.03,2157.11,6.94,2,139
O05231,2023-05-18,3,646.89,215.63,2157.11,3.75,2,133
O05319,2023-05-12,4,6731.64,1682.91,2157.11,39.01,2,28


In [0]:
if spark.catalog.tableExists("db_catalog.gold.fact_orders"):
    
    dlt_obj = DeltaTable.forName(spark, "db_catalog.gold.fact_orders")

    dlt_obj.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id = src.order_id AND trg.dim_customer_key = src.dim_customer_key AND trg.dim_product_key = src.dim_product_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    df_fact_new.write.format("delta")\
            .option("path","abfss://gold@dbproject.dfs.core.windows.net/fact_orders")\
            .saveAsTable("db_catalog.gold.fact_orders")

In [0]:
%sql
select * from db_catalog.gold.fact_orders

order_id,order_date,quantity,total_amount,unit_price,customer_avg_order_value,ptg_of_customer_spend_on_order,dim_customer_key,dim_product_key
O00186,2023-02-18,1,1431.12,1431.12,3315.17,10.79,1,126
O02770,2023-12-09,2,461.12,230.56,3315.17,3.48,1,19
O05531,2023-09-10,5,9524.35,1904.87,3315.17,71.82,1,456
O08359,2023-03-09,1,1844.1,1844.1,3315.17,13.91,1,338
O00775,2024-01-16,1,1248.81,1248.81,2157.11,7.24,2,323
O02495,2023-05-09,4,1158.68,289.67,2157.11,6.71,2,354
O04328,2024-07-18,5,61.75,12.35,2157.11,0.36,2,155
O04879,2023-06-20,3,1197.09,399.03,2157.11,6.94,2,139
O05231,2023-05-18,3,646.89,215.63,2157.11,3.75,2,133
O05319,2023-05-12,4,6731.64,1682.91,2157.11,39.01,2,28
